# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Top 10 real estate sales per ward in 2025**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-05-13 18:06:48 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-05-13T18:06:48.542246")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `real estate sales`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 5 datasets matching 'real estate sales'

1. **City-Owned Property (Real Estate Tax Database)**
   ID: `city-owned-property`
   This dataset contains a listing of property owned by the City of Pittsburgh obtained from the City's Real Estate Database. For a more complete listing of City-owned properties obta
   - City-Owned Properties (CSV) [DataStore] ID: `4ff5eb17-e2ad-4818-97c4-8f91fc6b6396`
   - Data Dictionary with Field Defintions (XLSX) [DataStore] ID: `9070809b-54d9-441d-802e-812fcb7c825a`

2. **City Treasury Sales**
   ID: `city-treasury-sales`
   A listing of all the properties c
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'real estate sales', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: Search for Datasets

**Search query:** `property sales transactions deed transfers ward`

**Result preview:**
```
Found 2 datasets matching 'property sales transactions deed transfers ward'

1. **Allegheny County Property Sale Transactions**
   ID: `real-estate-sales`
   This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.  Before doing any market analysis on property sales, check the sales va
   - Property Sales Transactions (CSV) [DataStore] ID: `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
   - Allegheny County Information Portal Real Estate Sales (HTML) ID: `bfdf3a78-7830-425d-afa2-8fbc99c79dcb`
   - Property Dashboard (HTML) ID: `637ab4db-1313-4512-9c2
```


In [ ]:
# Step 2: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'property sales transactions deed transfers ward', "rows": 10}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 3: Dataset Details

**Dataset:** `real-estate-sales`

**Result preview:**
```
# Allegheny County Property Sale Transactions

This dataset contains data on all Real Property parcels that have sold since 2013 in Allegheny County, PA.

Before doing any market analysis on property sales, check the sales validation codes. Many property "sales" are not considered a valid representation of the true market value of the property. For example, when multiple lots are together on one deed with one price they are generally coded as invalid ("H") because the sale price for each parcel ID number indicates the total price paid for a group of parcels, not just for one parcel. See the 
```


In [ ]:
# Step 3: Dataset Details

# Get dataset details
resp = requests.get("https://data.wprdc.org/api/3/action/package_show", params={"id": 'real-estate-sales'})
ds = resp.json()["result"]

print(f"Title: {ds['title']}")
print(f"Description: {ds.get('notes', 'N/A')[:200]}")
print(f"\nResources:")
for r in ds.get("resources", []):
    act = "DataStore" if r.get("datastore_active") else "File"
    print(f"  - {r['name']} ({r['format']}, {act}) ID: {r['id']}")


## Step 4: Load Data from Resource

**Resource ID:** `5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1`
**Limit:** 5

**Result preview:**
```
Resource: 5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1
Total records: 491,017
Loaded: 5
Fields (25): PARID, FULL_ADDRESS, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESSDIR, PROPERTYADDRESSSTREET, PROPERTYADDRESSSUF, PROPERTYADDRESSUNITDESC, PROPERTYUNITNO, PROPERTYCITY, PROPERTYSTATE, PROPERTYZIP, SCHOOLCODE, SCHOOLDESC, MUNICODE, MUNIDESC, RECORDDATE, SALEDATE, PRICE, DEEDBOOK, DEEDPAGE, SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC

Sample (5 rows):

           PARID                          FULL_ADDRESS PROPERTYHOUSENUM PROPERTYFRACTION PROPERTYADDRESSDIR PROPERTYADDRESSSTREET PROPERTYADDRESSSUF 
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 5: Load Data from Resource

**Resource ID:** `57b84145-f9eb-493b-a78f-9bc803413d99`
**Limit:** 5

**Result preview:**
```
Resource: 57b84145-f9eb-493b-a78f-9bc803413d99
Total records: 31,405
Loaded: 5
Fields (25): PARID, FULL_ADDRESS, PROPERTYHOUSENUM, PROPERTYFRACTION, PROPERTYADDRESSDIR, PROPERTYADDRESSSTREET, PROPERTYADDRESSSUF, PROPERTYADDRESSUNITDESC, PROPERTYUNITNO, PROPERTYCITY, PROPERTYSTATE, PROPERTYZIP, SCHOOLCODE, SCHOOLDESC, MUNICODE, MUNIDESC, RECORDDATE, SALEDATE, PRICE, DEEDBOOK, DEEDPAGE, SALECODE, SALEDESC, INSTRTYP, INSTRTYPDESC

Sample (5 rows):

           PARID                             FULL_ADDRESS PROPERTYHOUSENUM PROPERTYFRACTION PROPERTYADDRESSDIR PROPERTYADDRESSSTREET PROPERTYADDRESSSU
```


In [ ]:
# Step 5: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": '57b84145-f9eb-493b-a78f-9bc803413d99', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 6: SQL Analysis Query

**SQL:**
```sql

SELECT 
  "MUNIDESC" AS ward,
  COUNT(*) AS total_sales,
  SUM("PRICE") AS total_sale_value,
  ROUND(AVG("PRICE")::numeric, 0) AS avg_sale_price
FROM "57b84145-f9eb-493b-a78f-9bc803413d99"
WHERE "MUNIDESC" ILIKE '%PITTSBURGH%'
  AND "SALEDATE" >= '2025-01-01'
  AND "SALEDATE" <= '2025-12-31'
GROUP BY "MUNIDESC"
ORDER BY total_sales DESC
LIMIT 10

```

**Result preview:**
```
SQL: 
SELECT 
  "MUNIDESC" AS ward,
  COUNT(*) AS total_sales,
  SUM("PRICE") AS total_sale_value,
  ROUND(AVG("PRICE")::numeric, 0) AS avg_sale_price
FROM "57b84145-f9eb-493b-a78f-9bc803413d99"
WHERE "MUNIDESC" ILIKE '%PITTSBURGH%'
  AND "SALEDATE" >= '2025-01-01'
  AND "SALEDATE" <= '2025-12-31'
GROUP BY "MUNIDESC"
ORDER BY total_sales DESC
LIMIT 10

Rows: 10
Columns: ward, total_sales, total_sale_value, avg_sale_price

                  ward  total_sales  total_sale_value  avg_sale_price
19th Ward - PITTSBURGH          716       135605185.0          189393
14th Ward - PITTSBURGH          66
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = '\nSELECT \n  "MUNIDESC" AS ward,\n  COUNT(*) AS total_sales,\n  SUM("PRICE") AS total_sale_value,\n  ROUND(AVG("PRICE")::numeric, 0) AS avg_sale_price\nFROM "57b84145-f9eb-493b-a78f-9bc803413d99"\nWHERE "MUNIDESC" ILIKE \'%PITTSBURGH%\'\n  AND "SALEDATE" >= \'2025-01-01\'\n  AND "SALEDATE" <= \'2025-12-31\'\nGROUP BY "MUNIDESC"\nORDER BY total_sales DESC\nLIMIT 10\n'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here are the **Top 10 Pittsburgh Wards by Number of Real Estate Sales in 2025**:

---

## 🏠 Top 10 Real Estate Sales by Ward — Pittsburgh, 2025

| Rank | Ward | # of Sales | Total Sale Value | Avg Sale Price |
|------|------|----------:|----------------:|---------------:|
| 1 | **19th Ward** | 716 | $135,605,185 | $189,393 |
| 2 | **14th Ward** | 668 | $276,216,568 | $413,498 |
| 3 | **10th Ward** | 431 | $92,503,231 | $217,144 |
| 4 | **20th Ward** | 421 | $60,265,000 | $143,147 |
| 5 | **27th Ward** | 355 | $204,270,389 | $575,410 |
| 6 | **26th Ward** | 332 | $79,614,872 | $239,804 |
| 7 | **15th Ward** | 325 | $40,549,161 | $124,767 |
| 8 | **12th Ward** | 279 | $11,852,312 | $42,481 |
| 9 | **18th Ward** | 276 | $25,834,340 | $93,603 |
| 10 | **4th Ward** | 269 | $84,945,220 | $315,781 |

---

### 🔑 Key Highlights
- **19th Ward** (Brookline/Beechview area) leads with **716 sales** — the most active ward in Pittsburgh.
- **14th Ward** (Squirrel Hill South) ranks 2nd in volume but tops the list in **total value at $276M**, reflecting its higher-priced real estate market.
- **27th Ward** (Carrick/Mt. Oliver area) has the **highest average sale price at $575,410**, suggesting larger or more premium property transactions.
- **12th Ward** has the lowest average price ($42,481) among the top 10, indicating more affordable or distressed property sales.

### ⚠️ Caveats
- Sale prices include **all transaction types** (market sales, treasurer sales, love & affection transfers, etc.). For true market value analysis, filter by valid sale codes.
- Data sourced from the **Allegheny County Property Sale Transactions** dataset on the [WPRDC portal](https://data.wprdc.org), updated monthly.**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-05-13

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-05-13 18:06:48
- **Query**: Top 10 real estate sales per ward in 2025
- **Data Source**: WPRDC

---

*Generated by  AI Data Concierge v0.1.0*
